In [1]:
from embedder import Embedder

model = Embedder()

q1 = "How does approximate nearest neighbor search work?"

2026-06-24 11:00:26.275092411 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


## Q1

In [2]:
v1 = model.encode(q1)
print(v1[0])

-0.02058203437252893


## Q2

In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [7]:
c1 = [page['content'] for page in documents if page['filename'] == "02-vector-search/lessons/07-sqlitesearch-vector.md"][0]

In [8]:
v1.dot(model.encode(c1))

np.float64(0.36107026789538205)

## Q3

In [13]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

batch = [chunk['content'] for chunk in chunks]

X = model.encode_batch(batch)

scores = X.dot(v1)

In [21]:
import numpy as np
idx = np.argmax(scores)
idx, scores[idx], chunks[idx]['filename']

(np.int64(94),
 np.float64(0.6489016436447387),
 '02-vector-search/lessons/07-sqlitesearch-vector.md')

## Q4

In [22]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(X, chunks)

In [24]:
vindex.search(model.encode('What metric do we use to evaluate a search engine?'), num_results=1)

[{'start': 0,
  'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our set

## Q5

In [32]:
from minsearch import Index
txtindex = Index(text_fields=['content'], keyword_fields=['filename'])
txtindex.fit(chunks)

In [36]:
q2 = "How do I store vectors in PostgreSQL?"
txtres = txtindex.search(q2, num_results=5)
vecres = vindex.search(model.encode(q2), num_results=5)

set([res['filename'] for res in vecres]) - set([res['filename'] for res in txtres])

{'02-vector-search/lessons/08-pgvector.md'}

## Q6

In [37]:
NUM_RESULTS = 50
def rrf(result_lists, k=60, num_results=NUM_RESULTS):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

q3 = "How do I give the model access to tools?"
txtres = txtindex.search(q3, num_results=NUM_RESULTS)
vecres = vindex.search(model.encode(q3), num_results=NUM_RESULTS)
rrf_res = rrf([txtres, vecres])


In [44]:
[res['filename'] for res in rrf_res[:5]], [res['filename'] for res in vecres[:5]], [res['filename'] for res in txtres[:5]]


(['01-agentic-rag/lessons/13-function-calling.md',
  '01-agentic-rag/lessons/13-function-calling.md',
  '01-agentic-rag/lessons/13-function-calling.md',
  '01-agentic-rag/lessons/14-agentic-loop.md',
  '01-agentic-rag/lessons/01-intro.md'],
 ['01-agentic-rag/lessons/01-intro.md',
  '04-evaluation/lessons/02-ground-truth.md',
  '01-agentic-rag/lessons/16-other-frameworks.md',
  '01-agentic-rag/lessons/15-frameworks.md',
  '01-agentic-rag/lessons/13-function-calling.md'],
 ['01-agentic-rag/lessons/14-agentic-loop.md',
  '01-agentic-rag/lessons/13-function-calling.md',
  '01-agentic-rag/lessons/13-function-calling.md',
  '01-agentic-rag/lessons/13-function-calling.md',
  '04-evaluation/lessons/02-ground-truth.md'])